<a href="https://colab.research.google.com/github/jujubinha2003598-collab/POO/blob/main/C%C3%B3pia_de_Python_SQL_ORM_IFMG_Parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SQLAlchemy ORM Parte 2: Relacionamentos

Na parte 1 trabalhamos com **uma única tabela**. Na prática, os dados se relacionam: um livro tem uma editora, um livro tem autores, e assim por diante. Nesta aula vamos aprender a:

1. Modelar uma relação **um para muitos** (Editora → Livros)
2. Modelar uma relação **muitos para muitos** (Livros ↔ Autores)
3. Consultar várias tabelas de uma vez com **JOIN**

Na parte 1, o autor era um texto dentro da tabela `livros`. Isso repete dados (o nome "George Orwell" aparece em dois livros) e facilita erros de digitação. A solução do modelo relacional é separar os dados em tabelas e ligá-las por **chaves estrangeiras**.

### 1. Relação um para muitos (1:N)

Um **livro** tem **uma** editora, mas uma **editora** tem **vários** livros.

```
   editoras                          livros
 +----+----------------+        +----+---------------------+------+------------+
 | id | nome           |        | id | titulo              | ano  | editora_id |
 +----+----------------+        +----+---------------------+------+------------+
 |  1 | Cia das Letras |<-------|  1 | 1984                | 1949 |     1      |
 |    |                |<-------|  2 | Rev. dos Bichos     | 1945 |     1      |
 |  2 | Aleph          |<-------|  3 | Flores para Algernon| 1966 |     2      |
 +----+----------------+        +----+---------------------+------+------------+
```

> **Regra:** a chave estrangeira fica sempre no lado **"muitos"** (aqui, em `livros`).

No SQLAlchemy usamos duas peças diferentes:

| Peça | O que é | Existe no banco? |
|---|---|---|
| `ForeignKey('editoras.id')` | Coluna que guarda o `id` da editora | **Sim**, é uma coluna real |
| `relationship(...)` | Atributo Python que dá acesso ao(s) objeto(s) relacionado(s) | **Não**, não cria coluna nenhuma |


<br>

---

<br>

#### `relationship()` e `back_populates`

`relationship('Livro')` cria um atributo na classe que **navega** até os objetos da outra tabela, sem precisarmos escrever SQL. O SQLAlchemy usa a `ForeignKey` para descobrir como ligar as duas tabelas.

Declaramos o `relationship` **nas duas classes**, e o parâmetro `back_populates` diz o nome do atributo na *outra* ponta. Assim as duas pontas ficam sincronizadas:

| Classe | Atributo | Devolve |
|---|---|---|
| `Livro` | `livro.editora` | **um** objeto `Editora` |
| `Editora` | `editora.livros` | uma **lista** de objetos `Livro` |

Se fazemos `livro.editora = aleph`, o livro entra automaticamente em `aleph.livros` (e vice-versa).

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Table
from sqlalchemy import select, func, desc
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

### criando a conexão
engine = create_engine('sqlite:///livraria_rel.db')

### criando as tabelas
Base = declarative_base()
class Editora(Base):
    __tablename__ = 'editoras'
    id = Column(Integer, primary_key=True)
    nome = Column(String)

    # uma editora tem uma LISTA de livros
    livros = relationship('Livro', back_populates='editora')

    def __repr__(self):
        return f"Editora({self.id}, {self.nome!r})"

class Livro(Base):
    __tablename__ = 'livros'
    id = Column(Integer, primary_key=True)
    titulo = Column(String)
    ano = Column(Integer)
    editora_id = Column(Integer, ForeignKey('editoras.id'))  # coluna real (chave estrangeira)

    # um livro tem UMA editora
    editora = relationship('Editora', back_populates='livros')

    def __repr__(self):
        return f"Livro({self.id}, {self.titulo!r}, {self.ano})"

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

Note que `'Livro'` e `'Editora'` aparecem entre aspas dentro do `relationship`: é o nome da classe em texto, pois uma classe pode ser citada antes de ser definida.


<br>

---

<br>

#### Populando as tabelas
Podemos ligar os objetos de dois jeitos: pelo lado "muitos" (`editora=`) ou pelo lado "um" (`livros.append()`). Não precisamos mexer em `editora_id`: o SQLAlchemy preenche a chave estrangeira sozinho no `commit()`.

In [ ]:
Session = sessionmaker(bind=engine)
session = Session()

# criando editoras
cia = Editora(nome='Companhia das Letras')
aleph = Editora(nome='Aleph')
principis = Editora(nome='Principis')

# jeito 1 de relacionar: o livro aponta para a sua editora
cia_1984 = Livro(titulo='1984', ano=1949, editora=cia)
cia_bichos = Livro(titulo='A Revolução dos Bichos', ano=1945, editora=cia)

# jeito 2 de relacionar: a editora recebe os seus livros
aleph.livros.append(Livro(titulo='Flores para Algernon', ano=1966))
principis.livros = [
    Livro(titulo='Dom Casmurro', ano=1899),
    Livro(titulo='Memórias Póstumas de Brás Cubas', ano=1881),
]

# back_populates em ação: ainda ANTES do commit, os dois lados já estão sincronizados
# (o id aparece como None porque só é gerado pelo banco no commit)
print("cia", cia.livros)
print("aleph", aleph.livros)
print("principis", principis.livros)

# Ao adicionar as editoras, os livros ligados a elas entram na sessão junto
session.add_all([cia, aleph, principis])
session.commit()

cia [Livro(None, '1984', 1949), Livro(None, 'A Revolução dos Bichos', 1945)]
aleph [Livro(None, 'Flores para Algernon', 1966)]
principis [Livro(None, 'Dom Casmurro', 1899), Livro(None, 'Memórias Póstumas de Brás Cubas', 1881)]


#### Consultando
Note no resultado da consulta a seguir que ao adicionar as editoras no banco, o SQLAlchemy também populou a tabela de livros.

In [ ]:
# De editora para livros (1 -> N): editora.livros é uma lista
print("----- Editoras -----")
for linha in session.execute(select(Editora)).all():
    print(linha.Editora.nome, '\t->\t', [livro.titulo for livro in linha.Editora.livros])

print()

# De livro para editora (N -> 1): livro.editora é um único objeto
print("----- Livros -----")
for linha in session.execute(select(Livro)).all():
    print(linha.Livro.titulo, '\t->\t', linha.Livro.editora.nome)

----- Editoras -----
Companhia das Letras 	->	 ['1984', 'A Revolução dos Bichos']
Aleph 	->	 ['Flores para Algernon']
Principis 	->	 ['Dom Casmurro', 'Memórias Póstumas de Brás Cubas']
Companhia das Letras 	->	 ['1984', 'A Revolução dos Bichos']
Aleph 	->	 ['Flores para Algernon']
Principis 	->	 ['Dom Casmurro', 'Memórias Póstumas de Brás Cubas']

----- Livros -----
1984 	->	 Companhia das Letras
A Revolução dos Bichos 	->	 Companhia das Letras
Flores para Algernon 	->	 Aleph
Dom Casmurro 	->	 Principis
Memórias Póstumas de Brás Cubas 	->	 Principis
1984 	->	 Companhia das Letras
A Revolução dos Bichos 	->	 Companhia das Letras
Flores para Algernon 	->	 Aleph
Dom Casmurro 	->	 Principis
Memórias Póstumas de Brás Cubas 	->	 Principis


In [ ]:
session.close()

### 2. Relação muitos para muitos (N:N)

Agora vamos criar a tabela **autores** para substituir a coluna de texto `autor`. Um livro pode ter **vários autores** (ex.: *Belas Maldições*, de Neil Gaiman e Terry Pratchett) e um autor pode ter **vários livros**.

Uma única chave estrangeira não resolve: em qual tabela ela ficaria? A solução é uma **tabela de associação**, que guarda apenas pares `(livro, autor)`:

```
   livros                livros_autores               autores
 +----+------------+    +----------+----------+    +----+----------------+
 | id | titulo     |    | livro_id | autor_id |    | id | nome           |
 +----+------------+    +----------+----------+    +----+----------------+
 |  1 | 1984       |<---|    1     |    1     |--->|  1 | George Orwell  |
 |  6 | Belas Mald.|<---|    6     |    4     |--->|  4 | Neil Gaiman    |
 |    |            |<---|    6     |    5     |--->|  5 | Terry Pratchett|
 +----+------------+    +----------+----------+    +----+----------------+
```

Cada linha da associação é uma ligação. A chave primária da associação é a **combinação** das duas colunas, o que impede repetir o mesmo par.

<br>

---

<br>

#### O parâmetro `secondary`
Em um `relationship` muitos-para-muitos, `secondary=` informa **por qual tabela de associação** o SQLAlchemy deve passar para ligar as duas classes. Com ele, `livro.autores` vira uma lista de `Autor`, e `autor.livros` uma lista de `Livro`, e o SQLAlchemy cuida sozinho das linhas da associação: ao fazer `livro.autores.append(autor)`, ele insere o par em `livros_autores` no `commit()`.

Repare que:
* a tabela de associação é um objeto `Table`, **sem classe própria**, pois só guarda as duas chaves estrangeiras;
* o `back_populates` funciona como antes, nas duas classes, tornando a relação **bidirecional**.

Vamos reescrever o modelo completo, agora com `Autor` (e sem a coluna `autor` de texto), em um novo banco:

In [ ]:
engine = create_engine('sqlite:///livraria_completa.db')
Base = declarative_base()   # novo Base: o modelo abaixo substitui o anterior

# Tabela de associação (N:N entre livros e autores)
livros_autores = Table(
    'livros_autores', Base.metadata,
    Column('livro_id', Integer, ForeignKey('livros.id'), primary_key=True),
    Column('autor_id', Integer, ForeignKey('autores.id'), primary_key=True),
)

class Editora(Base):
    __tablename__ = 'editoras'
    id = Column(Integer, primary_key=True)
    nome = Column(String)
    livros = relationship('Livro', back_populates='editora')              # 1:N

    def __repr__(self):
        return f"Editora({self.id}, {self.nome!r})"

class Autor(Base):
    __tablename__ = 'autores'
    id = Column(Integer, primary_key=True)
    nome = Column(String)
    livros = relationship('Livro', secondary=livros_autores,              # N:N
                          back_populates='autores')

    def __repr__(self):
        return f"Autor({self.id}, {self.nome!r})"

class Livro(Base):
    __tablename__ = 'livros'
    id = Column(Integer, primary_key=True)
    titulo = Column(String)
    ano = Column(Integer)

    # relação com Editora 1:N -> guarda a chave estrangeira aqui
    editora_id = Column(Integer, ForeignKey('editoras.id'))
    editora = relationship('Editora', back_populates='livros')            # N:1

    # relação com Autor N:N -> chaves estrangeiras na tabela de associação
    autores = relationship('Autor', secondary=livros_autores,             # N:N
                           back_populates='livros')

    def __repr__(self):
        return f"Livro({self.id}, {self.titulo!r}, {self.ano})"

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

#### Populando as tabelas
Para o livro com dois autores, basta passar os dois na lista `autores`. Também criamos a editora *Rocco*, que não tem nenhum livro (vamos usá-la nos JOINs).

In [ ]:
Session = sessionmaker(bind=engine)
session = Session()

# Editoras (a Rocco ficará sem livros de propósito)
cia = Editora(nome='Companhia das Letras')
aleph = Editora(nome='Aleph')
principis = Editora(nome='Principis')
intrinseca = Editora(nome='Intrínseca')
rocco = Editora(nome='Rocco')

# Autores
orwell = Autor(nome='George Orwell')
keyes = Autor(nome='Daniel Keyes')
machado = Autor(nome='Machado de Assis')
gaiman = Autor(nome='Neil Gaiman')
pratchett = Autor(nome='Terry Pratchett')

# Livros: cada um aponta para UMA editora e recebe uma LISTA de autores
livros = [
    Livro(titulo='1984', ano=1949, editora=cia, autores=[orwell]),
    Livro(titulo='A Revolução dos Bichos', ano=1945, editora=cia, autores=[orwell]),
    Livro(titulo='Flores para Algernon', ano=1966, editora=aleph, autores=[keyes]),
    Livro(titulo='Dom Casmurro', ano=1899, editora=principis, autores=[machado]),
    Livro(titulo='Memórias Póstumas de Brás Cubas', ano=1881, editora=principis, autores=[machado]),
    Livro(titulo='Belas Maldições', ano=1990, editora=intrinseca, autores=[gaiman, pratchett]),
]

# Os livros arrastam editoras e autores para a sessão;
session.add_all(livros)
# só a Rocco (sem livros) precisa ser adicionada
session.add(rocco)
session.commit()

#### Navegando nas duas direções
Como a relação é bidirecional, podemos partir de qualquer lado. Ao adicionar um item em uma ponta, a outra ponta é atualizada (`back_populates`):

In [ ]:
print("Livros do Orwell:      ", orwell.livros)
print("Autores de Belas Maldições:", livros[5].autores)
print("Livros do Pratchett:   ", pratchett.livros)

# Adicionando pelo lado do autor: o livro também "enxerga" o autor
deuses = Livro(titulo='Deuses Americanos', ano=2001, editora=intrinseca)
session.add(deuses)
gaiman.livros.append(deuses)
print("\nAutores de Deuses Americanos:", deuses.autores)

session.commit()

Livros do Orwell:       [Livro(1, '1984', 1949), Livro(2, 'A Revolução dos Bichos', 1945)]
Autores de Belas Maldições: [Autor(4, 'Neil Gaiman'), Autor(5, 'Terry Pratchett')]
Livros do Pratchett:    [Livro(6, 'Belas Maldições', 1990)]

Autores de Deuses Americanos: [Autor(4, 'Neil Gaiman')]


Por baixo dos panos, o SQLAlchemy gravou cada par na tabela de associação. Podemos consultá-la diretamente, como qualquer tabela:

In [ ]:
for linha in session.execute(select(livros_autores)):
    print(linha.livro_id, linha.autor_id)

6 5
1 1
2 1
3 2
4 3
5 3
6 4
7 4


### 3. Consultas com JOIN

Com mais de uma tabela, uma consulta pode precisar de **colunas de tabelas diferentes** (ex.: o título do livro *e* o nome da editora), ou filtrar uma tabela por uma coluna de outra. Para isso usamos o `JOIN`.

No SQLAlchemy, `.join()` recebe o atributo `relationship`, e o SQLAlchemy monta a condição `ON` sozinho, a partir das chaves estrangeiras.

| Operação | SQL | SQLAlchemy |
|---|---|---|
| JOIN (1:N) | `SELECT livros.titulo, editoras.nome FROM livros JOIN editoras ON editoras.id = livros.editora_id;` | `select(Livro.titulo, Editora.nome).join(Livro.editora)` |
| JOIN com filtro | `... WHERE editoras.nome = 'Principis';` | `.where(Editora.nome == 'Principis')` |
| JOIN (N:N) | `SELECT livros.titulo, autores.nome FROM livros JOIN livros_autores ON livros.id = livros_autores.livro_id JOIN autores ON autores.id = livros_autores.autor_id;` | `select(Livro.titulo, Autor.nome).join(Livro.autores)` |
| JOIN + agrupamento | `SELECT editoras.nome, COUNT(livros.id) FROM editoras JOIN livros ON ... GROUP BY editoras.nome;` | `select(Editora.nome, func.count(Livro.id)).join(Editora.livros).group_by(Editora.nome)` |
| LEFT JOIN | `... FROM editoras LEFT OUTER JOIN livros ON ...` | `.join(Editora.livros, isouter=True)` |

#### JOIN entre `livros` e `editoras`
Cada linha do resultado combina um livro com a sua editora. Compare o SQL gerado com a tabela acima:

In [ ]:
# SELECT livros.titulo, editoras.nome FROM livros JOIN editoras ON editoras.id = livros.editora_id
query = select(Livro.titulo, Editora.nome).join(Livro.editora)

print(query)
print()
for linha in session.execute(query):
    print(linha.titulo, '-', linha.nome)

SELECT livros.titulo, editoras.nome 
FROM livros JOIN editoras ON editoras.id = livros.editora_id

1984 - Companhia das Letras
A Revolução dos Bichos - Companhia das Letras
Flores para Algernon - Aleph
Dom Casmurro - Principis
Memórias Póstumas de Brás Cubas - Principis
Belas Maldições - Intrínseca
Deuses Americanos - Intrínseca


*Também* podemos escrever a condição `ON` à mão, passando a tabela e a condição. O resultado é o mesmo, mas é mais útil quando não há `relationship` pronto:

```python
select(Livro.titulo, Editora.nome).join(Editora, Livro.editora_id == Editora.id)
```

#### JOIN com filtro
O JOIN permite filtrar uma tabela por colunas de outra. Quais livros são da editora Principis?

In [ ]:
query = (
    select(Livro.titulo, Livro.ano)
    .join(Livro.editora)
    .where(Editora.nome == 'Principis')
)

print(query)
print()
for linha in session.execute(query):
    print(linha.ano, linha.titulo)

SELECT livros.titulo, livros.ano 
FROM livros JOIN editoras ON editoras.id = livros.editora_id 
WHERE editoras.nome = :nome_1

1899 Dom Casmurro
1881 Memórias Póstumas de Brás Cubas


#### JOIN em relação muitos-para-muitos
Basta fazer `.join(Livro.autores)`. O SQLAlchemy percebe o `secondary` e gera **dois JOINs**: de `livros` para a tabela de associação e dela para `autores`. Um livro com dois autores aparece em duas linhas.


In [ ]:
query = select(Livro.titulo, Autor.nome).join(Livro.autores)

print(query)
print()
for titulo, autor in session.execute(query):
    print(f"{titulo} - {autor}")

SELECT livros.titulo, autores.nome 
FROM livros JOIN livros_autores AS livros_autores_1 ON livros.id = livros_autores_1.livro_id JOIN autores ON autores.id = livros_autores_1.autor_id

Belas Maldições - Terry Pratchett
1984 - George Orwell
A Revolução dos Bichos - George Orwell
Flores para Algernon - Daniel Keyes
Dom Casmurro - Machado de Assis
Memórias Póstumas de Brás Cubas - Machado de Assis
Belas Maldições - Neil Gaiman
Deuses Americanos - Neil Gaiman


Também é possível **encadear** vários `.join()` para juntar três tabelas. Título, editora e autor de cada livro:

In [ ]:
query = (
    select(Livro.titulo, Editora.nome, Autor.nome)
    .join(Livro.editora)
    .join(Livro.autores)
    .order_by(Livro.titulo)
)

for titulo, editora, autor in session.execute(query):
    print(f"{titulo} | {editora} | {autor}")

1984 | Companhia das Letras | George Orwell
A Revolução dos Bichos | Companhia das Letras | George Orwell
Belas Maldições | Intrínseca | Terry Pratchett
Belas Maldições | Intrínseca | Neil Gaiman
Deuses Americanos | Intrínseca | Neil Gaiman
Dom Casmurro | Principis | Machado de Assis
Flores para Algernon | Aleph | Daniel Keyes
Memórias Póstumas de Brás Cubas | Principis | Machado de Assis


> **Navegar ou fazer JOIN?** Navegar (`livro.editora.nome`) é prático para trabalhar com um objeto por vez, mas dispara um `SELECT` a cada acesso. O JOIN busca tudo em **uma** consulta, e é a escolha certa para relatórios, filtros e agregações.

In [ ]:
session.close()

### Exercício

Vamos modelar uma **escola** com três tabelas: **cursos**, **alunos** e **disciplinas**.

* Um **curso** tem vários **alunos**, e cada aluno pertence a **um** curso (relação **1:N**).
* Um **aluno** cursa várias **disciplinas**, e uma **disciplina** tem vários alunos matriculados (relação **N:N**, bidirecional).

**1. Modelagem.** Crie um novo `Base` e as classes abaixo (com `__repr__`, se quiser). Os dois relacionamentos devem ser bidirecionais, com `back_populates`:
* `Curso` (tabela `cursos`): `id` e `nome`. Atributo `alunos`.
* `Aluno` (tabela `alunos`): `id`, `nome` e `curso_id` (chave estrangeira). Atributos `curso` e `disciplinas`.
* `Disciplina` (tabela `disciplinas`): `id` e `nome`. Atributo `alunos`.
* Tabela de associação `matriculas`, com `aluno_id` e `disciplina_id`.

**2. Criação.** Use um banco novo (`sqlite:///escola.db`) e crie as tabelas com `Base.metadata.create_all(engine)`.

**3. Inserção.** Insira os dados abaixo e faça o `commit()`:

| Curso | | Disciplina | |
|---|---|---|---|
| Informática | | Banco de Dados | |
| Eletrônica | | Programação | |
| | | Circuitos | |
| | | Matemática | |

| Aluno | Curso | Disciplinas matriculadas |
|---|---|---|
| Ana | Informática | Banco de Dados, Programação, Matemática |
| Bruno | Informática | Programação, Matemática |
| Carla | Eletrônica | Circuitos, Matemática |
| Diego | Eletrônica | Circuitos |
| Elisa | Informática | Banco de Dados, Programação, Matemática |

**4. Consultas com JOIN.** Escreva e execute:
* a) O `nome` de cada aluno junto com o `nome` do seu curso (JOIN 1:N).
* b) O `nome` dos alunos matriculados em "Banco de Dados" (JOIN N:N com filtro). *Resultado esperado: Ana e Elisa.*
* c) **Desafio:** a quantidade de alunos matriculados em cada disciplina, da maior para a menor (JOIN N:N com `group_by`). *Resultado esperado: Matemática 4, Programação 3, e Banco de Dados e Circuitos com 2 cada (empate).*

> **Dica:** você vai precisar de `ForeignKey`, `Table` e `relationship`, que se importam de `sqlalchemy` (os dois primeiros) e de `sqlalchemy.orm` (o último).

In [ ]:
cursor.execute("""
CREATE TABLE aluno (
    id_aluno INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE disciplina (
    id_disciplina INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE matricula (
    id_aluno INTEGER,
    id_disciplina INTEGER,
    FOREIGN KEY (id_aluno) REFERENCES aluno(id_aluno),
    FOREIGN KEY (id_disciplina) REFERENCES disciplina(id_disciplina)
);
""")

In [ ]:
cursor.execute("INSERT INTO aluno (nome) VALUES ('Lucas Mendes');")
cursor.execute("INSERT INTO aluno (nome) VALUES ('Beatriz Costa');")
cursor.execute("INSERT INTO aluno (nome) VALUES ('Rafael Rocha');")


cursor.execute("INSERT INTO disciplina (nome) VALUES ('Banco de Dados');")
cursor.execute("INSERT INTO disciplina (nome) VALUES ('Programação Python');")
cursor.execute("INSERT INTO disciplina (nome) VALUES ('Redes de Computadores');")

cursor.execute("INSERT INTO matricula (id_aluno, id_disciplina) VALUES (1, 1);")
cursor.execute("INSERT INTO matricula (id_aluno, id_disciplina) VALUES (1, 2);")
cursor.execute("INSERT INTO matricula (id_aluno, id_disciplina) VALUES (2, 1);")
cursor.execute("INSERT INTO matricula (id_aluno, id_disciplina) VALUES (3, 3);")

connection.commit()

In [ ]:
print("=== 4a. Relação de Alunos e suas Disciplinas ===")
cursor.execute("""
SELECT aluno.nome, disciplina.nome
FROM matricula
JOIN aluno ON matricula.id_aluno = aluno.id_aluno
JOIN disciplina ON matricula.id_disciplina = disciplina.id_disciplina;
""")
for linha in cursor.fetchall():
    print(f"Aluno: {linha[0]} | Disciplina: {linha[1]}")

In [ ]:
print("\n=== 4b. Alunos matriculados em 'Banco de Dados' ===")
cursor.execute("""
SELECT aluno.nome
FROM matricula
JOIN aluno ON matricula.id_aluno = aluno.id_aluno
JOIN disciplina ON matricula.id_disciplina = disciplina.id_disciplina
WHERE disciplina.nome = 'Banco de Dados';
""")
for linha in cursor.fetchall():
    print(f"Aluno: {linha[0]}")

In [ ]:
print("\n=== 4c. Quantidade de Alunos por Disciplina ===")
cursor.execute("""
SELECT disciplina.nome, COUNT(matricula.id_aluno) AS total_alunos
FROM disciplina
LEFT JOIN matricula ON disciplina.id_disciplina = matricula.id_disciplina
GROUP BY disciplina.id_disciplina, disciplina.nome;
""")
for linha in cursor.fetchall():
    print(f"Disciplina: {linha[0]} | Total de Alunos: {linha[1]}")

connection.close()